# Marginal MCMC — PROMIS COPD (Factorized GRM)

Runs NUTS on item parameters with abilities Rao-Blackwellized out
for three model variants: baseline, pairwise imputation, and mixed imputation.
Data from Harvard Dataverse (doi:10.7910/DVN/UOQNJF).

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import os, sys
os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['JAX_ENABLE_X64'] = '1'
os.environ['TQDM_DISABLE'] = '1'
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

## 1. Load Data

In [ ]:
from bayesianquilts.data.promis_copd import get_multidomain_data, item_keys, response_cardinality

df, num_people, scale_indices = get_multidomain_data(polars_out=True, min_items=10)
print(f"Dataset: {num_people} people, {len(item_keys)} items, {len(scale_indices)} domains")
for domain, indices in scale_indices.items():
    print(f"  {domain}: {len(indices)} items")

def make_data_dict(dataframe):
    data = {}
    for col in dataframe.columns:
        data[col] = dataframe[col].to_numpy().astype(np.float64)
    data['person'] = np.arange(len(dataframe), dtype=np.float64)
    return data

base_data = make_data_dict(df)

BATCH_SIZE = 256
def data_factory():
    indices = np.arange(num_people)
    np.random.shuffle(indices)
    for start in range(0, num_people, BATCH_SIZE):
        end = min(start + BATCH_SIZE, num_people)
        yield {k: v[indices[start:end]] for k, v in base_data.items()}

## 2. Load Pairwise Stacking Model

In [ ]:
from bayesianquilts.irt.factorizedgrm import FactorizedGRModel
from pathlib import Path
from bayesianquilts.imputation.pairwise_stacking import PairwiseOrdinalStackingModel

pairwise_model = None
pairwise_path = Path('pairwise_stacking_model.yaml')
if pairwise_path.exists():
    pairwise_model = PairwiseOrdinalStackingModel.load(str(pairwise_path))
    print(f"Loaded pairwise stacking model: {len(pairwise_model.variable_names)} variables")
else:
    print("No pairwise stacking model found — only baseline variant will run")

## 3. MCMC Helper

In [ ]:
NUM_CHAINS = 2
NUM_WARMUP = 500
NUM_SAMPLES = 200
STEP_SIZE = 0.001

def run_variant(variant_name, model, data, seed):
    print(f"\n=== {variant_name.upper()} ===")
    mcmc_samples = model.fit_marginal_mcmc(
        data,
        theta_grid=None,
        num_chains=NUM_CHAINS,
        num_warmup=NUM_WARMUP,
        num_samples=NUM_SAMPLES,
        target_accept_prob=0.85,
        step_size=STEP_SIZE,
        seed=seed,
        verbose=True,
    )

    eap_result = model.compute_eap_abilities(data)
    eap = np.array(eap_result['eap'])
    print(f"  EAP: mean={np.mean(eap):.4f}, std={np.std(eap):.4f}")

    stats = model.standardize_marginal(data)
    eap_result = model.compute_eap_abilities(data)
    eap_std = np.array(eap_result['eap'])
    print(f"  Post-std EAP: mean={np.mean(eap_std):.4f}, std={np.std(eap_std):.4f}")

    model.fit_surrogate_to_mcmc()
    eap_arr = np.array(eap_result['eap'])
    model.surrogate_sample['abilities'] = jnp.array(
        eap_arr[:, np.newaxis, np.newaxis, np.newaxis]
    )[np.newaxis, ...]

    model.save_to_disk(f'grm_mcmc_{variant_name}')

    os.makedirs('mcmc_samples', exist_ok=True)
    save_dict = {}
    for var_name, samples in mcmc_samples.items():
        save_dict[var_name] = np.array(samples)
        flat = np.array(samples).reshape(-1, *samples.shape[2:])
        print(f"  {var_name}: mean={np.mean(flat):.4f}, std={np.std(flat):.4f}")
        if samples.shape[0] > 1:
            chain_means = np.mean(np.array(samples), axis=1)
            between_var = np.var(chain_means, axis=0, ddof=1)
            within_var = np.mean(np.var(np.array(samples), axis=1, ddof=1), axis=0)
            n = samples.shape[1]
            r_hat = np.sqrt(((n - 1) / n * within_var + between_var) / np.maximum(within_var, 1e-30))
            print(f"    R-hat: mean={np.mean(r_hat):.4f}, max={np.max(r_hat):.4f}")
    save_dict['eap'] = eap
    save_dict['eap_standardized'] = eap_std
    save_dict['psd'] = np.array(eap_result['psd'])
    save_dict['standardize_mu'] = stats['mu']
    save_dict['standardize_sigma'] = stats['sigma']
    np.savez(f'mcmc_samples/mcmc_{variant_name}.npz', **save_dict)

    return eap_std, np.array(eap_result['psd'])

## 4. Baseline MCMC (No Imputation)

In [ ]:
model = FactorizedGRModel.load_from_disk('grm_baseline')
data = dict(base_data)
eap_baseline, psd_baseline = run_variant('baseline', model, data, seed=42)
del model; gc.collect()

## 5. Pairwise MCMC (Pairwise Stacking Imputation)

In [ ]:
if pairwise_model is not None:
    model = FactorizedGRModel.load_from_disk('grm_baseline')
    model.imputation_model = pairwise_model
    data = dict(base_data)
    pmfs, weights = model._compute_batch_pmfs(data)
    if pmfs is not None:
        data['_imputation_pmfs'] = pmfs
    print("Pairwise imputation PMFs attached")
    eap_pairwise, psd_pairwise = run_variant('pairwise', model, data, seed=43)
    del model; gc.collect()
else:
    print("Skipping — no pairwise model available")
    eap_pairwise = psd_pairwise = None

## 6. Mixed MCMC (Pairwise + IRT Imputation)

In [ ]:
if pairwise_model is not None:
    from bayesianquilts.imputation.mixed import IrtMixedImputationModel
    model = FactorizedGRModel.load_from_disk('grm_baseline')

    surrogate = model.surrogate_distribution_generator(model.params)
    samples = surrogate.sample(32, seed=jax.random.PRNGKey(142))
    model.surrogate_sample = samples
    model.calibrated_expectations = {
        k: jnp.mean(v, axis=0) for k, v in samples.items()
    }

    mixed_model = IrtMixedImputationModel(
        irt_model=model,
        mice_model=pairwise_model,
        data_factory=data_factory,
        irt_elpd_batch_size=4,
    )
    model.imputation_model = mixed_model

    data = dict(base_data)
    pmfs, weights = model._compute_batch_pmfs(data)
    if pmfs is not None:
        data['_imputation_pmfs'] = pmfs
        if weights is not None:
            data['_imputation_weights'] = weights
    print("Mixed imputation PMFs attached")
    eap_mixed, psd_mixed = run_variant('mixed', model, data, seed=44)
    del model, mixed_model; gc.collect()
else:
    print("Skipping — no pairwise model available")
    eap_mixed = psd_mixed = None

## 7. Compare Variants

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# EAP distributions
for eap, label in [(eap_baseline, 'Baseline'), (eap_pairwise, 'Pairwise'), (eap_mixed, 'Mixed')]:
    if eap is not None:
        axes[0].hist(eap, bins=50, density=True, alpha=0.5, label=label)
axes[0].set_title('EAP Ability Distributions')
axes[0].set_xlabel('θ')
axes[0].legend()

# Baseline vs Pairwise scatter
if eap_pairwise is not None:
    axes[1].scatter(eap_baseline, eap_pairwise, alpha=0.2, s=3)
    axes[1].plot([-3, 3], [-3, 3], 'r--', alpha=0.5)
    axes[1].set_xlabel('Baseline θ')
    axes[1].set_ylabel('Pairwise θ')
    axes[1].set_title('Baseline vs Pairwise')

# Baseline vs Mixed scatter
if eap_mixed is not None:
    axes[2].scatter(eap_baseline, eap_mixed, alpha=0.2, s=3)
    axes[2].plot([-3, 3], [-3, 3], 'r--', alpha=0.5)
    axes[2].set_xlabel('Baseline θ')
    axes[2].set_ylabel('Mixed θ')
    axes[2].set_title('Baseline vs Mixed')

fig.suptitle('PROMIS COPD — MCMC Variant Comparison')
fig.tight_layout()
fig.savefig('mcmc_variant_comparison.pdf', bbox_inches='tight', dpi=150)
plt.show()

## Summary

Ran marginal MCMC (NUTS with Rao-Blackwellized abilities) for three
model variants on PROMIS COPD:
- **Baseline**: ignorable missingness
- **Pairwise**: pairwise ordinal stacking imputation
- **Mixed**: pairwise + IRT blended imputation

4 chains × 500 samples each.